# validation-no-grad — ex2: @torch.no_grad() decorator on the validation function — same effect, cleaner API

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `validation-no-grad`. Running the final beacon cell reports progress against the `PyTorch: no_grad validation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: no_grad validation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`validation-no-grad`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "validation-no-grad"
DD_SUBTOPIC = "PyTorch: no_grad validation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `@torch.no_grad()` decorator vs the `with torch.no_grad():` block

Ex1 wrapped the validation loop body in `with torch.no_grad():`. The deepening move: use `@torch.no_grad()` as a DECORATOR on the eval function itself. Same effect, cleaner API.

```python
@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_correct = 0
    total = 0
    for x, y in loader:
        out = model(x)
        total_correct += (out.argmax(dim=1) == y).sum().item()
        total += y.numel()
    return total_correct / total
```

**Why the decorator is preferred.** Wraps the ENTIRE function body, including the loop setup, the iteration, and any helpers called inline. The `with` block can leak grad-enabled paths through helper calls if the helper itself opens a `with torch.enable_grad()`. The decorator makes the no-grad contract part of the function's signature.

**Still need `.eval()`.** `no_grad` only disables autograd; it does NOT change Dropout/BN mode. Both calls are required at inference.

### Exercise 2 — @torch.no_grad() decorator on the validation function — same effect, cleaner API

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `@torch.no_grad()` as a function decorator on a validation routine and confirm autograd is disabled inside the function body even when the caller has autograd enabled.
> Keywords: no_grad, decorator, validation, inference
> ```

**KCs targeted:** `no-grad-decorator-vs-context`, `eval-mode-plus-no-grad-pairing`

Implement `ex2_validate(model, x, y)` decorated with `@torch.no_grad()`. The decorator-style equivalent of ex1's `with torch.no_grad():` block.

Inputs:
- `model`: an `nn.Module` (will be put into eval mode).
- `x`: input tensor.
- `y`: integer label tensor of shape `(B,)`.

Algorithm (inside the decorated function):
1. `model.eval()` — flip to eval mode (Dropout off, BN running stats frozen).
2. `logits = model(x)`.
3. `preds = logits.argmax(dim=1)`.
4. `acc = (preds == y).float().mean().item()`.
5. Inside the function (before returning), assert `not torch.is_grad_enabled()`. This is the proof that the decorator is in effect.
6. Return a `dict` with:
   - `'logits'`: the logits tensor.
   - `'preds'`: the prediction tensor.
   - `'acc'`: the Python float accuracy.
   - `'grad_enabled_inside'`: `False` (read from `torch.is_grad_enabled()` before returning).

Constraint: do NOT use `with torch.no_grad():` inside the function. The decorator is the whole point.

In [ ]:
import torch

@torch.no_grad()
def ex2_validate(model, x: Tensor, y: Tensor) -> dict:
    """Decorator-style no-grad validation: eval + argmax + accuracy."""
    raise NotImplementedError()


def _test_ex2():
    import torch

    # === Build a tiny classifier ===
    t.manual_seed(0)
    model = nn.Sequential(
        nn.Linear(8, 16),
        nn.ReLU(),
        nn.Dropout(p=0.5),
        nn.Linear(16, 4),
    )
    x = t.randn(20, 8)
    y = t.randint(0, 4, (20,))

    # === Confirm caller has autograd ON before calling ex2_validate ===
    assert torch.is_grad_enabled(), 'test setup: caller should have grad enabled'

    # === Run the validation ===
    out = ex2_validate(model, x, y)
    assert set(out.keys()) == {'logits', 'preds', 'acc', 'grad_enabled_inside'}, (
        f'keys wrong: {set(out.keys())}'
    )

    # === Decorator was in effect inside the function ===
    assert out['grad_enabled_inside'] is False, (
        f'@torch.no_grad() must disable grad inside the body; got {out["grad_enabled_inside"]}'
    )

    # === Grad re-enabled after the function returns (decorator scope) ===
    assert torch.is_grad_enabled(), 'decorator must restore caller grad state'

    # === No grad metadata on the returned logits ===
    assert out['logits'].requires_grad is False, (
        f'logits computed under no_grad must not require grad; got requires_grad={out["logits"].requires_grad}'
    )
    assert out['logits'].grad_fn is None, (
        f'logits must have no grad_fn under no_grad; got {out["logits"].grad_fn}'
    )

    # === Model is in eval mode after the call ===
    assert not model.training, 'model.eval() should have been called inside ex2_validate'

    # === Shapes + types ===
    assert out['logits'].shape == (20, 4)
    assert out['preds'].shape == (20,)
    assert out['preds'].dtype == t.long, f'argmax must give long, got {out["preds"].dtype}'
    assert isinstance(out['acc'], float), f'acc must be Python float, got {type(out["acc"]).__name__}'
    assert 0.0 <= out['acc'] <= 1.0, f'acc out of range: {out["acc"]}'

    # === Dropout disabled in eval: two passes give identical logits ===
    model.train()  # flip back to train; ex2_validate should still .eval() internally
    out_a = ex2_validate(model, x, y)
    out_b = ex2_validate(model, x, y)
    assert t.allclose(out_a['logits'], out_b['logits']), (
        'Two validation passes must give identical logits (model.eval() disables Dropout)'
    )

    # === Matches a hand-written `with torch.no_grad():` equivalent ===
    model.train()
    out_dec = ex2_validate(model, x, y)
    model.eval()
    with torch.no_grad():
        logits_ref = model(x)
        preds_ref = logits_ref.argmax(dim=1)
        acc_ref = (preds_ref == y).float().mean().item()
    assert t.allclose(out_dec['logits'], logits_ref), 'decorator-form must match with-block form'
    assert t.equal(out_dec['preds'], preds_ref)
    assert abs(out_dec['acc'] - acc_ref) < 1e-6
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
import torch

@torch.no_grad()
def ex2_validate(model, x, y):
    model.eval()
    logits = model(x)
    preds = logits.argmax(dim=1)
    acc = (preds == y).float().mean().item()
    grad_inside = torch.is_grad_enabled()
    return {
        'logits': logits,
        'preds': preds,
        'acc': float(acc),
        'grad_enabled_inside': bool(grad_inside),
    }
```

**Decorator vs context manager — same effect, different scope.** `@torch.no_grad()` toggles grad off for the whole function body and restores the prior state on return. A `with` block toggles only for the lexical block. For an entire eval function, the decorator is more readable and harder to accidentally bypass.

**`.eval()` is still required.** `no_grad` disables autograd; it does NOT switch Dropout or BatchNorm modes. Forgetting `.eval()` would give you autograd-free outputs that are STILL stochastic (Dropout) or use BATCH stats (BN).

**Why `grad_enabled_inside` is a useful return value.** It's an introspection probe — the test asserts it's False without needing to inspect the function's bytecode. In production code, this kind of probe is overkill, but here it makes the decorator's effect testable from the outside.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()